# 📺 YouTube Data Analysis

This notebook presents a comprehensive analysis of YouTube data across two datasets:
- **UScomments.csv** — ~691,000 US YouTube comments
- **Trending video CSVs** — ~340,000 trending video records from 10+ countries (US, UK, CA, MX, IN, DE, JP, KR, FR, RU)

The analysis is divided into four main parts:

| # | Section | Description |
|---|---------|-------------|
| 1 | 💬 Sentiment Analysis | Score every comment using NLTK VADER |
| 2 | ☁️ Word Cloud Analysis | Visualize top words in positive & negative comments |
| 3 | 😂 Emoji Analysis | Extract and rank the most-used emojis |
| 4 | 📊 Trending Video Analysis | Engagement rates, top channels, correlations, and more |

---
**Author:** Priscila Salgado  
**Email:** priscisb09@gmail.com  
**LinkedIn:** [priscila-salgado-84914615b](https://www.linkedin.com/in/priscila-salgado-84914615b)

---
## ⚙️ Section 0 — Imports & Setup

All required libraries are imported here. Run this cell before any other section.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objs as go
from plotly.offline import iplot
import os
import string
import warnings
from warnings import filterwarnings
from collections import Counter
from sqlalchemy import create_engine

import nltk
nltk.download("vader_lexicon", quiet=True)
from nltk.sentiment.vader import SentimentIntensityAnalyzer

!pip install wordcloud emoji==2.14.1 -q
from wordcloud import WordCloud, STOPWORDS
import emoji

filterwarnings('ignore')
print("✅ All libraries loaded successfully.")

---
## 💬 Section 1 — Comment Sentiment Analysis

In this section we load the US YouTube comments dataset, clean it, and apply **NLTK's VADER** (Valence Aware Dictionary and sEntiment Reasoner) to assign a polarity score to each comment.

**Polarity score range:**
- `+1.0` → Most positive
- `0.0` → Neutral
- `-1.0` → Most negative

### 1.1 Load & Explore the Comments Dataset

In [ ]:
comments = pd.read_csv(
    "/Users/priscilasalgado/Desktop/Proyect 1/UScomments.csv",
    engine="python",
    on_bad_lines="skip"
)

print(f"Dataset shape: {comments.shape}")
print(f"Columns: {list(comments.columns)}")
comments.head(10)

### 1.2 Data Cleaning

Check for and remove null values before proceeding with the analysis.

In [ ]:
# Check for missing values
print("Missing values before cleaning:")
print(comments.isnull().sum())

# Drop rows with missing comment text
comments.dropna(inplace=True)

print(f"\nMissing values after cleaning:")
print(comments.isnull().sum())
print(f"\nClean dataset shape: {comments.shape}")

### 1.3 Sentiment Scoring with VADER

We initialize the `SentimentIntensityAnalyzer` and compute a **compound polarity score** for every comment in the dataset.

In [ ]:
sia = SentimentIntensityAnalyzer()

# Test the scorer on a sample comment
test_comment = "I love this video so much!"
score = sia.polarity_scores(test_comment)['compound']
print(f"Sample comment: '{test_comment}'")
print(f"Polarity score: {score}")

In [ ]:
# Apply sentiment scoring to all comments
comments['polarity'] = comments['comment_text'].astype(str).apply(
    lambda x: sia.polarity_scores(x)['compound']
)

print("Polarity score distribution:")
print(comments['polarity'].describe())
comments.head(5)

### 1.4 Segment Comments by Sentiment

We split comments into **highly positive** (polarity ≥ 0.8) and **highly negative** (polarity ≤ -0.8) groups for deeper analysis.

In [ ]:
# Filter highly positive comments
filter_pos = (comments['polarity'] >= 0.8) & (comments['polarity'] <= 1.0)
comments_positive = comments[filter_pos]

# Filter highly negative comments
filter_neg = (comments['polarity'] >= -1.0) & (comments['polarity'] <= -0.8)
comments_negative = comments[filter_neg]

print(f"✅ Highly positive comments : {comments_positive.shape[0]:,}")
print(f"❌ Highly negative comments : {comments_negative.shape[0]:,}")
print(f"📊 Positive-to-negative ratio: {comments_positive.shape[0] / comments_negative.shape[0]:.1f}x")

---
## ☁️ Section 2 — Word Cloud Analysis

Word clouds help visualize the most frequent words in each sentiment group. Common stopwords (like "the", "and", "I") are filtered out to focus on meaningful terms.

### 2.1 Positive Comments Word Cloud

In [ ]:
total_positive_comments = ' '.join(comments_positive['comment_text'])

wordcloud_positive = WordCloud(
    stopwords=set(STOPWORDS),
    background_color='white',
    colormap='Greens',
    width=800, height=400
).generate(total_positive_comments)

plt.figure(figsize=(12, 6))
plt.imshow(wordcloud_positive, interpolation='bilinear')
plt.axis('off')
plt.title('Most Frequent Words in Highly Positive Comments', fontsize=16, pad=15)
plt.tight_layout()
plt.show()

### 2.2 Negative Comments Word Cloud

In [ ]:
total_negative_comments = ' '.join(comments_negative['comment_text'])

wordcloud_negative = WordCloud(
    stopwords=set(STOPWORDS),
    background_color='white',
    colormap='Reds',
    width=800, height=400
).generate(total_negative_comments)

plt.figure(figsize=(12, 6))
plt.imshow(wordcloud_negative, interpolation='bilinear')
plt.axis('off')
plt.title('Most Frequent Words in Highly Negative Comments', fontsize=16, pad=15)
plt.tight_layout()
plt.show()

---
## 😂 Section 3 — Emoji Analysis

Here we extract every emoji used across all 691K comments and identify the **Top 10 most-used emojis**.

### 3.1 Extract All Emojis from Comments

In [ ]:
all_emojis_found = []

for comment in comments['comment_text']:
    emojis_info = emoji.emoji_list(str(comment))
    emojis_found = [item['emoji'] for item in emojis_info]
    all_emojis_found.extend(emojis_found)

print(f"Total emojis found across all comments: {len(all_emojis_found):,}")
print(f"Sample: {all_emojis_found[0:10]}")

### 3.2 Top 10 Most-Used Emojis

In [ ]:
emojis_count_list_top10 = Counter(all_emojis_found).most_common(10)

emojis_list = [e for e, count in emojis_count_list_top10]
counts = [count for e, count in emojis_count_list_top10]

print("Top 10 emojis:")
for e, c in emojis_count_list_top10:
    print(f"  {e}  —  {c:,}")

In [ ]:
trace = go.Bar(
    x=emojis_list,
    y=counts,
    marker_color='steelblue',
    text=counts,
    textposition='outside'
)

layout = go.Layout(
    title='Top 10 Most-Used Emojis in YouTube Comments',
    xaxis=dict(title='Emoji'),
    yaxis=dict(title='Count'),
    plot_bgcolor='white'
)

iplot(go.Figure(data=[trace], layout=layout))

---
## 📊 Section 4 — Trending Video Analysis (Multi-Country)

In this section we analyze trending video metadata from **10 countries**: US, UK, CA, MX, IN, DE, JP, KR, FR, and RU.  
The dataset includes views, likes, dislikes, comment counts, channel names, categories, and more.

### 4.1 Load & Merge Multi-Country Data

In [ ]:
path = r'/Users/priscilasalgado/Desktop/Proyect 1/additional_data'
files = os.listdir(path)
files_csv = [f for f in files if f.endswith('.csv')]

print(f"CSV files found: {files_csv}")

full_df = pd.DataFrame()

for file in files_csv:
    current_df = pd.read_csv(
        os.path.join(path, file),
        encoding='iso-8859-1',
        on_bad_lines='skip'
    )
    full_df = pd.concat([full_df, current_df], ignore_index=True)

print(f"\nCombined dataset shape: {full_df.shape}")

### 4.2 Data Cleaning — Remove Duplicates

In [ ]:
print(f"Rows before deduplication : {full_df.shape[0]:,}")
print(f"Duplicate rows found       : {full_df[full_df.duplicated()].shape[0]:,}")

full_df = full_df.drop_duplicates()

print(f"Rows after deduplication   : {full_df.shape[0]:,}")
full_df.head(3)

### 4.3 Export Sample to SQLite Database

We export a 1,000-row sample to a local SQLite database for SQL-based querying.

In [ ]:
full_df[0:1000].to_csv(
    r'/Users/priscilasalgado/Desktop/Proyect 1/youtube_sample.csv',
    index=False
)

engine = create_engine(r'sqlite:////Users/priscilasalgado/Desktop/Proyect 1/youtube_sample.sqlite')
full_df[0:1000].to_sql('Videos', con=engine, if_exists='replace', index=False)

print("✅ Sample exported to CSV and SQLite successfully.")

### 4.4 Category Mapping

The dataset uses numeric `category_id` values. We map these to readable category names using YouTube's official JSON category files.

In [ ]:
json_df = pd.read_json(
    r'/Users/priscilasalgado/Desktop/Proyect 1/additional_data/US_category_id.json'
)

cat_dict = {}
for item in json_df['items'].values:
    cat_dict[int(item['id'])] = [item['snippet']['title']]

full_df['category_name'] = full_df['category_id'].map(cat_dict)
full_df_exploded = full_df.explode('category_name')

print("Category mapping applied.")
print(f"Unique categories: {full_df_exploded['category_name'].nunique()}")
full_df[['category_id', 'category_name']].drop_duplicates().sort_values('category_id').head(10)

### 4.5 Engagement Rate Feature Engineering

We compute three engagement rate metrics as a percentage of total views:
- **Like rate** = likes / views × 100
- **Dislike rate** = dislikes / views × 100
- **Comment rate** = comment_count / views × 100

In [ ]:
full_df['like_rate']    = (full_df['likes']         / full_df['views']) * 100
full_df['dislike_rate'] = (full_df['dislikes']      / full_df['views']) * 100
full_df['comment_rate'] = (full_df['comment_count'] / full_df['views']) * 100

full_df_exploded = full_df.explode('category_name')

print("Engagement rate summary:")
full_df[['like_rate', 'dislike_rate', 'comment_rate']].describe().round(4)

### 4.6 Like & Dislike Rate by Category

Box plots show the distribution of like and dislike rates across video categories, ordered by median like rate.

In [ ]:
order_like = (
    full_df_exploded
    .groupby('category_name')['like_rate']
    .median()
    .sort_values(ascending=False)
    .index
)

plt.figure(figsize=(14, 6))
sns.boxplot(x='category_name', y='like_rate', data=full_df_exploded,
            order=order_like, palette='Set2')
plt.xticks(rotation=90)
plt.xlabel('Category', fontsize=12)
plt.ylabel('Like Rate (%)', fontsize=12)
plt.title('Like Rate by Video Category (ordered by median)', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
order_dislike = (
    full_df_exploded
    .groupby('category_name')['dislike_rate']
    .median()
    .sort_values(ascending=False)
    .index
)

plt.figure(figsize=(14, 6))
sns.boxplot(x='category_name', y='dislike_rate', data=full_df_exploded,
            order=order_dislike, palette='Set3')
plt.xticks(rotation=90)
plt.xlabel('Category', fontsize=12)
plt.ylabel('Dislike Rate (%)', fontsize=12)
plt.title('Dislike Rate by Video Category (ordered by median)', fontsize=14)
plt.tight_layout()
plt.show()

### 4.7 Correlation Analysis — Views, Likes & Dislikes

How strongly are views, likes, and dislikes related to each other?

In [ ]:
print("Correlation Matrix:")
print(full_df[['views', 'likes', 'dislikes']].corr().round(3))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.regplot(x='views', y='likes', data=full_df, ax=axes[0],
            scatter_kws={'alpha': 0.3, 'color': 'steelblue'},
            line_kws={'color': 'red'})
axes[0].set_title('Views vs. Likes', fontsize=13)
axes[0].set_xlabel('Views')
axes[0].set_ylabel('Likes')

sns.heatmap(full_df[['views', 'likes', 'dislikes']].corr(),
            annot=True, fmt='.2f', cmap='coolwarm', ax=axes[1])
axes[1].set_title('Correlation Heatmap', fontsize=13)

plt.tight_layout()
plt.show()

### 4.8 Top 20 Trending Channels

Which channels appeared most frequently in the trending lists across all countries?

In [ ]:
cdf = (full_df
       .groupby('channel_title')
       .size()
       .sort_values(ascending=False)
       .reset_index()
       .rename(columns={0: 'total_videos'})
)

fig = px.bar(
    cdf[0:20],
    x='channel_title',
    y='total_videos',
    title='Top 20 Most Frequently Trending Channels',
    labels={'channel_title': 'Channel', 'total_videos': 'Trending Appearances'},
    color='total_videos',
    color_continuous_scale='Blues'
)
fig.update_layout(xaxis_tickangle=-45)
fig.show()

### 4.9 Title Punctuation vs. Views

Does using more punctuation in a video title affect view count?  
We count the number of punctuation characters in each title and compare against view performance.

In [ ]:
def punc_count(text):
    return len([char for char in str(text) if char in string.punctuation])

sample = full_df[0:10000].copy()
sample['count_punc'] = sample['title'].apply(punc_count)

plt.figure(figsize=(12, 6))
sns.boxplot(x='count_punc', y='views', data=sample, palette='Set2')
plt.xticks(rotation=90)
plt.xlabel('Number of Punctuation Characters in Title', fontsize=12)
plt.ylabel('Views', fontsize=12)
plt.title('Views by Punctuation Count in Video Title', fontsize=14)
plt.tight_layout()
plt.show()

---
## 📌 Key Findings

| # | Finding |
|---|---------|
| 1 | 😂 is the most-used emoji across all US YouTube comments with **36,998 occurrences** |
| 2 | Positive comments outnumber highly negative ones by nearly **4:1** (64K vs. 16K) |
| 3 | Views and likes show a **strong positive correlation (r ≈ 0.78)** |
| 4 | Dislikes have a weaker relationship with views (r ≈ 0.41) |
| 5 | **The Late Show with Stephen Colbert** had the most trending videos globally |
| 6 | Music and Entertainment categories consistently show the highest like rates |

---
*This project is for educational and portfolio purposes only.*